# System Accuracy Evaluation

This study compares system accuracy against a **ground truth** test set.

## Hybrid evaluation
1. **Keyword match** (fast, no API required)  
    - Does the response contain the expected keyword?

2. **LLM judge (Gemini)**  
    - Does the response semantically match the expected answer?


In [1]:
from __future__ import annotations

import os
from google import genai
from dotenv import load_dotenv

import sys
from pathlib import Path
base_path = Path.cwd().parent
sys.path.insert(0, str(base_path))

from preprocessor import DocumentPreprocessor
from vector_store import VectorStore
from agents.reformer import ReformerAgent
from agents.retriever import RetrieverAgent
from agents.validator import ValidatorAgent
from controller import Controller

d:\Desktop\doc_extraction_agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
    )

TEST_EXAMPLES = Path.cwd() / "test_examples"
PDF_PATH      = TEST_EXAMPLES / "attention_is_all_you_need.pdf"
DOC_ID        = "attention"

### EVALUATION METHODS

In [3]:
def keyword_match(answer: str, keywords: list[str]) -> bool:
    answer_lower = answer.lower()
    return any(kw.lower() in answer_lower for kw in keywords)


def llm_judge(question: str, expected: str, answer: str) -> bool:
    prompt = (
        f"Question: {question}\n"
        f"Expected answer: {expected}\n"
        f"System answer: {answer}\n\n"
        f"Does the system answer contain the same information as the expected answer? "
        f"Reply with only YES or NO."
    )
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    return "YES" in response.text.strip().upper()


def evaluate_one(question: str, expected: str, keywords: list[str], answer: str) -> tuple[bool, str]:
    if keyword_match(answer, keywords):
        return True, "keyword"
    if llm_judge(question, expected, answer):
        return True, "llm-judge"
    return False, "llm-judge"

## TESTS

In [4]:
store = VectorStore()
if not store.load(DOC_ID):
    print("Index not found, processing document")
    pre = DocumentPreprocessor()
    chunks, _ = pre.preprocess_pdf(PDF_PATH)
    store.add(chunks)
    store.save(DOC_ID)

controller = Controller(
    reformer  = ReformerAgent(),
    retriever = RetrieverAgent(vector_store=store),
    validator = ValidatorAgent(),
)


### TEST 1

In [5]:
case1={
        "question": "How many encoder layers does the Transformer have?",
        "expected": "6 encoder layers",
        "keywords": ["six"],
    }

In [6]:
result = controller.run(case1["question"])
is_correct, method = evaluate_one(
    case1["question"], case1["expected"], case1["keywords"], result.answer
)

status = "CORRECT" if is_correct else "WRONG"
print(f"    Question : {case1['question']}")
print(f"    Expected : {case1['expected']}")
print(f"    Answer   : {result.answer[:120]}")
print(f"    Result   : {status}  ({method})\n")


    Question : How many encoder layers does the Transformer have?
    Expected : 6 encoder layers
    Answer   : The Transformer has N = 6 encoder layers.
    Result   : CORRECT  (llm-judge)



### TEST 2

In [7]:
case2={
        "question": "What is the dimension of the model (d_model)?",
        "expected": "512",
        "keywords": ["512"],
    }

In [8]:
result = controller.run(case2["question"])
is_correct, method = evaluate_one(
    case2["question"], case2["expected"], case2["keywords"], result.answer
)

status = "CORRECT" if is_correct else "WRONG"
print(f"    Question : {case2['question']}")
print(f"    Expected : {case2['expected']}")
print(f"    Answer   : {result.answer[:120]}")
print(f"    Result   : {status}  ({method})\n")


    Question : What is the dimension of the model (d_model)?
    Expected : 512
    Answer   : The dimension of the model (dmodel) is 512.
    Result   : CORRECT  (keyword)



### TEST 3

In [9]:
case3={
        "question": "What type of attention mechanism does the Transformer use?",
        "expected": "Scaled dot-product attention (and multi-head attention)",
        "keywords": ["scaled dot-product", "multi-head", "self-attention"],
    }

In [10]:
result = controller.run(case3["question"])
is_correct, method = evaluate_one(
    case3["question"], case3["expected"], case3["keywords"], result.answer
)

status = "CORRECT" if is_correct else "WRONG"
print(f"    Question : {case3['question']}")
print(f"    Expected : {case3['expected']}")
print(f"    Answer   : {result.answer[:120]}")
print(f"    Result   : {status}  ({method})\n")


    Question : What type of attention mechanism does the Transformer use?
    Expected : Scaled dot-product attention (and multi-head attention)
    Answer   : The Transformer uses multi-headed self-attention. It also uses multi-head attention and relies entirely on self-attentio
    Result   : CORRECT  (keyword)

